> **Paper work:** prefer [`march18_paper_candidates.ipynb`](march18_paper_candidates.ipynb) and `malca.review.paper_candidates` for canonical migrated paths and full feature coverage.

# DustyCult Reviewed Dippers

Run DustyCult fits for the reviewed March 18 dipper candidates in `output/runs/runs_march18_bundle_all/review/review.taxonomy_filled.db`.

This notebook uses the existing `malca.review.dustycult` integration so fit metadata and posterior predictive curves are written back to the review DB tables used by the review app:

- `dustycult_fits`
- `dustycult_predictive_curves`
- `output/runs/runs_march18_bundle_all/review/dustycult/<candidate>/<mode>/`


## Setup

Path discovery lets this run from the repo root or from a notebook subdirectory. The execution cell later in the notebook writes DustyCult fit rows back to the review DB when you run it.

In [1]:
from __future__ import annotations

import json
import math
import sqlite3
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from scipy.optimize import least_squares

root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        p for p in root_candidates
        if (p / 'pyproject.toml').exists()
        and (p / 'malca' / 'review' / 'dustycult.py').exists()
    ),
    Path.cwd(),
)

for path in (REPO_ROOT, REPO_ROOT / 'malca'):
    text = str(path.resolve())
    if text not in sys.path:
        sys.path.insert(0, text)

from malca.io.notebook_paths import resolve_local_lightcurve_path
from malca.review.dustycult import (
    DUSTYCULT_BANDPASS_NM,
    check_dustycult_available,
    control_defaults_for_candidate,
    load_canonical_cleaned_lightcurve,
    load_dustycult_curve,
    load_dustycult_fits,
    prepare_dustycult_input,
    preferred_fit_mode,
    run_dustycult_fit,
)
from malca.review.dustycult_display import (
    build_dustycult_fit_figure,
    dustycult_fit_metadata_rows,
    dustycult_geometry_rows,
    dustycult_posterior_rows,
    select_dustycult_display_row,
)
from malca.review.dustycult_visualization import build_dustycult_occulter_figure
from malca.review.store import db_connect
from malca.stv.events import score_lightcurve

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 240)


/opt/homebrew/Caskroom/miniconda/base/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Edit these values before the execution cell if you want a smoke test or a forced rerun. The default run skips existing successful fits and retries failed or missing fits.

In [2]:
from malca.config import DEFAULT_OUTPUT_DIR
from malca.review.paper_candidates import MARCH18_RUN, REVIEW_DB, LIGHTCURVE_DIR as PAPER_LIGHTCURVE_DIR

RUN_DIR = REPO_ROOT / MARCH18_RUN
DB_PATH = REPO_ROOT / REVIEW_DB
LIGHTCURVE_DIR = REPO_ROOT / PAPER_LIGHTCURVE_DIR
RESULTS_DIR = RUN_DIR / 'results'

MODES = ('quick', 'full')
RERUN_EXISTING = False
MAX_CANDIDATES = None  # Set to 1 for a smoke test before running all 21.
JULIA = 'julia'
DUSTYCULT_PROJECT = REPO_ROOT / 'external' / 'dustycult'
EXPECTED_REVIEWED_DIPPERS = 21
RUN_DETERMINISTIC_PREFIT = True
DETERMINISTIC_GRID_N = 41
DETERMINISTIC_MAX_NFEV = 400
EXPORT_DETERMINISTIC_RESULTS = False
FULL_PLOT_MODE = 'full'
MAX_FIT_PLOTS = 21

WINDOW_STRATEGY = 'adaptive_best_local'
WINDOW_LOCAL_RADIUS_DAYS = 365
WINDOW_MIN_POINTS = 30
WINDOW_MIN_SIDE_POINTS = 5
WINDOW_MIN_HALF_WIDTH_DAYS = 7
WINDOW_MAX_HALF_WIDTH_DAYS = 365
WINDOW_PADDING_MIN_DAYS = 14
WINDOW_PADDING_MAX_DAYS = 180
WINDOW_BASELINE_SHOULDER_POINTS = 10
WINDOW_BASELINE_SHOULDER_MIN_DAYS = 30
WINDOW_BASELINE_FLUX_TOL = 0.015
WINDOW_BASELINE_RETURN_FRACTION = 0.10
WINDOW_EVENT_EXTENT_FRACTION = 0.20
WINDOW_EVENT_CLUSTER_GAP_DAYS = 90
WINDOW_SHOULDER_EXTRA_DAYS = 7
RERUN_ON_WINDOW_CHANGE = True
WINDOW_OVERRIDES = {}

RUN_PARAMS_PATH = RUN_DIR / 'run_params.json'
RUN_PARAMS = json.loads(RUN_PARAMS_PATH.read_text()) if RUN_PARAMS_PATH.exists() else {}

print(f'REPO_ROOT        = {REPO_ROOT}')
print(f'RUN_DIR          = {RUN_DIR}')
print(f'DB_PATH          = {DB_PATH}')
print(f'LIGHTCURVE_DIR   = {LIGHTCURVE_DIR}')
print(f'DUSTYCULT_PROJECT= {DUSTYCULT_PROJECT}')
print(f'DB exists        = {DB_PATH.exists()}')
print(f'run_params exists= {RUN_PARAMS_PATH.exists()}')

if not DB_PATH.exists():
    raise FileNotFoundError(DB_PATH)


REPO_ROOT        = /Users/calder/code/malca
RUN_DIR          = /Users/calder/code/malca/output/runs/runs_march18_bundle_all
DB_PATH          = /Users/calder/code/malca/output/runs/runs_march18_bundle_all/review/review.taxonomy_filled.db
LIGHTCURVE_DIR   = /Users/calder/code/malca/output/runs/runs_march18_bundle_all/bundle_assets/lightcurves
DUSTYCULT_PROJECT= /Users/calder/code/malca/external/dustycult
DB exists        = True
run_params exists= False


## DustyCult Availability

This verifies that Julia and the local `external/dustycult` project are available before the fit queue is built.

In [3]:
availability = check_dustycult_available(project_path=DUSTYCULT_PROJECT, julia=JULIA)
display(Markdown(f'**DustyCult availability:** `{availability.message}`'))
print(f'Julia executable: {availability.julia}')
print(f'Project path:     {availability.project_path}')
print(f'CLI script:       {availability.script_path}')

if not availability.ok:
    raise RuntimeError(availability.message)


**DustyCult availability:** `DustyCult is available`

Julia executable: julia
Project path:     /Users/calder/code/malca/external/dustycult
CLI script:       /Users/calder/code/malca/external/dustycult/scripts/fit_lightcurve.jl


## Load Reviewed Dippers

The predicate intentionally mirrors the review-app taxonomy labels and legacy direct class labels.

In [4]:
DIPPER_WHERE = '''
    r.event_class = 'dipper'
    OR r.morphology_primary = 'dimming_event'
    OR COALESCE(r.physical_secondary, '') LIKE '%dipper%'
    OR COALESCE(r.priority_tags_json, '') LIKE '%dipper%'
    OR COALESCE(r.morphology_secondary_json, '') LIKE '%dip%'
'''

with db_connect(DB_PATH) as conn:
    reviewed_dippers = pd.read_sql_query(
        f'''
        SELECT
            c.*,
            r.event_class,
            r.workflow_status,
            r.disposition,
            r.morphology_primary,
            r.morphology_secondary,
            r.morphology_secondary_json,
            r.physical_primary,
            r.physical_secondary,
            r.classification_confidence,
            r.priority_tags_json,
            r.notes AS review_notes,
            r.updated_at AS review_updated_at
        FROM reviews r
        JOIN candidates c USING(candidate_id)
        WHERE {DIPPER_WHERE}
        ORDER BY r.candidate_id
        ''',
        conn,
    )
    existing_fits = pd.read_sql_query(
        '''
        SELECT candidate_id, mode, status, updated_at, runtime_sec, t0_jd, start_jd, end_jd,
               n_input_points, n_curve_points, artifact_dir, error
        FROM dustycult_fits
        WHERE candidate_id IN (
            SELECT r.candidate_id FROM reviews r WHERE ''' + DIPPER_WHERE + '''
        )
        ORDER BY candidate_id, mode
        ''',
        conn,
    )

if len(reviewed_dippers) != EXPECTED_REVIEWED_DIPPERS:
    raise AssertionError(
        f'Expected {EXPECTED_REVIEWED_DIPPERS} reviewed dippers, found {len(reviewed_dippers)}. '
        'Inspect DIPPER_WHERE or the review DB before running fits.'
    )

print(f'Reviewed dippers: {len(reviewed_dippers)}')
display(reviewed_dippers[['candidate_id', 'event_class', 'morphology_secondary', 'physical_primary', 'physical_secondary', 'disposition', 'dipper_score', 'dipper_n_valid_dips', 'dip_run_count', 'dip_best_morph']])
display(existing_fits if not existing_fits.empty else Markdown('No existing DustyCult fits for these reviewed dippers.'))


Reviewed dippers: 21


,candidate_id,event_class,morphology_secondary,physical_primary,physical_secondary,disposition,dipper_score,dipper_n_valid_dips,dip_run_count,dip_best_morph
0,111669557747,dipper,None,None,None,keep,19.914586,15.0,3.0,gaussian
1,146029419304,dipper,None,None,None,keep,21.255792,31.0,3.0,gaussian
2,180388903123,dipper,None,None,None,keep,-1.040463,4.0,4.0,gaussian
3,206158525635,dipper,None,None,None,keep,20.882040,87.0,1.0,skew_gaussian
4,223338997633,dipper,None,None,None,keep,19.852360,30.0,3.0,gaussian
5,240518636016,dipper,None,None,None,keep,19.800610,41.0,2.0,gaussian
6,249109084130,dipper,None,None,None,keep,0.000000,0.0,0.0,none
7,25771086021,dipper,None,None,None,keep,15.589737,26.0,1.0,gaussian
8,369367489518,dipper,None,None,None,keep,21.709102,42.0,5.0,gaussian
9,446676921101,dipper,None,None,None,keep,-0.560155,28.0,1.0,gaussian


,candidate_id,mode,status,updated_at,runtime_sec,t0_jd,start_jd,end_jd,n_input_points,n_curve_points,artifact_dir,error
0,111669557747,quick,ok,2026-06-02T22:19:06.154767+00:00,40.590364,8569.134531,8449.134531,8689.134531,50,50,/Users/calder/code/malca/output/runs/runs_marc...,
1,549755992463,quick,ok,2026-06-01T16:57:03.169597+00:00,NaN,9994.795326,9874.795326,10114.795326,99,99,/Users/calder/code/malca/output/runs/runs_marc...,
2,661425468910,quick,failed,2026-05-26T22:06:54.962502+00:00,2.309656,9654.071633,9649.631410,9655.632190,7,0,/Users/calder/code/malca/output/runs/runs_marc...,DustyCult exited with status 1


## Candidate Helpers

These helpers localize stale light-curve paths from transferred DB payloads and decide whether a fit mode should be skipped or retried.

In [5]:
def _finite_float(value, default=None):
    try:
        if value is None or pd.isna(value):
            return default
    except Exception:
        if value is None:
            return default
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if math.isfinite(number) else default


def clean_value(value):
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def row_payload(row: pd.Series) -> dict[str, object]:
    payload = {}
    raw = row.get('payload_json')
    if isinstance(raw, str) and raw.strip():
        try:
            payload.update(json.loads(raw))
        except Exception:
            pass
    for key, value in row.items():
        cleaned = clean_value(value)
        if cleaned is not None:
            payload[key] = cleaned
    payload['candidate_id'] = str(row['candidate_id'])
    return payload


def resolve_candidate_lc_path(row: pd.Series, payload: dict[str, object] | None = None) -> Path | None:
    payload = payload or row_payload(row)
    candidates = [
        row.get('lc_path'),
        payload.get('lc_path'),
        payload.get('path'),
        payload.get('dat_path'),
        payload.get('asas_sn_id'),
        row.get('candidate_id'),
    ]
    for value in candidates:
        if value is None:
            continue
        resolved = resolve_local_lightcurve_path(value, run_dir=RUN_DIR, repo_root=REPO_ROOT)
        if resolved is not None and resolved.exists():
            return resolved
    return None


def mode_status(fits: pd.DataFrame, candidate_id: str, mode: str) -> str | None:
    if fits.empty:
        return None
    sub = fits[(fits['candidate_id'].astype(str) == str(candidate_id)) & (fits['mode'].astype(str) == mode)]
    if sub.empty:
        return None
    return str(sub.iloc[-1].get('status') or '') or None


WINDOW_MATCH_TOL_DAYS = 1e-3


def latest_mode_fit_row(fits: pd.DataFrame, candidate_id: str, mode: str, *, require_ok: bool = False) -> pd.Series | None:
    if fits.empty:
        return None
    sub = fits[(fits['candidate_id'].astype(str) == str(candidate_id)) & (fits['mode'].astype(str) == str(mode))].copy()
    if require_ok:
        sub = sub[sub['status'].astype(str).eq('ok')]
    if sub.empty:
        return None
    if 'updated_at' in sub.columns:
        sub = sub.sort_values('updated_at')
    return sub.iloc[-1]


def fit_window_matches(fits: pd.DataFrame, candidate_id: str, mode: str, controls: dict[str, object] | None, *, tol_days: float = WINDOW_MATCH_TOL_DAYS) -> bool:
    if not controls:
        return False
    row = latest_mode_fit_row(fits, candidate_id, mode, require_ok=True)
    if row is None:
        return False
    for row_key, control_key in (('t0_jd', 't0_jd'), ('start_jd', 'start_jd'), ('end_jd', 'end_jd')):
        current = _finite_float(controls.get(control_key))
        stored = _finite_float(row.get(row_key)) if row_key in row.index else None
        if current is None or stored is None or abs(current - stored) > tol_days:
            return False
    return True


def should_run_mode(fits: pd.DataFrame, candidate_id: str, mode: str, controls: dict[str, object] | None = None) -> bool:
    status = mode_status(fits, candidate_id, mode)
    if RERUN_EXISTING or status != 'ok':
        return True
    if RERUN_ON_WINDOW_CHANGE and controls is not None:
        return not fit_window_matches(fits, candidate_id, mode, controls)
    return False


def planned_mode_action(fits: pd.DataFrame, candidate_id: str, mode: str, controls: dict[str, object] | None = None) -> str:
    status = mode_status(fits, candidate_id, mode)
    if RERUN_EXISTING:
        return 'rerun_forced'
    if status != 'ok':
        return 'run_missing_or_failed'
    if RERUN_ON_WINDOW_CHANGE and controls is not None and not fit_window_matches(fits, candidate_id, mode, controls):
        return 'rerun_window_changed'
    return 'skipped_existing_ok'


def require_adaptive_windows_ready(candidate_ids) -> None:
    if WINDOW_STRATEGY != 'adaptive_best_local':
        return
    cached = globals().get('adaptive_controls_by_candidate', {})
    missing = [str(candidate_id) for candidate_id in candidate_ids if str(candidate_id) not in cached]
    if missing:
        preview = ', '.join(missing[:5])
        raise RuntimeError(f'Run the Window Diagnostics cell before fitting; missing adaptive windows for {preview}.')


def active_controls_for_candidate(
    conn,
    candidate_id: str,
    payload: dict[str, object],
    *,
    lc_path: Path | None = None,
    run_params: dict | None = None,
) -> dict[str, object]:
    cached = globals().get('adaptive_controls_by_candidate', {})
    if WINDOW_STRATEGY == 'adaptive_best_local' and str(candidate_id) in cached:
        return dict(cached[str(candidate_id)])
    return control_defaults_for_candidate(
        conn,
        candidate_id,
        payload,
        lc_path=lc_path,
        run_params=run_params,
    )


def selected_candidate_frame() -> pd.DataFrame:
    data = reviewed_dippers.copy()
    if MAX_CANDIDATES is not None:
        data = data.head(int(MAX_CANDIDATES)).copy()
    return data.reset_index(drop=True)


## Window Diagnostics

Build the adaptive DustyCult fit windows before any deterministic or MCMC fits run. This cell is read-only with respect to the review DB; later fit cells use the cached `adaptive_controls_by_candidate` values produced here.


In [6]:
def _event_scoring_kwargs(run_params: dict | None) -> tuple[object, dict[str, object], list[str], dict[str, object]]:
    from malca.config import (
        LOGBF_THRESHOLD_DIP,
        LOGBF_THRESHOLD_JUMP,
        MAG_POINTS,
        P_POINTS,
        RUN_MAX_GAP_POINTS,
        RUN_MIN_POINTS,
        SIGNIFICANCE_THRESHOLD,
        TRIGGER_MODE,
    )
    from malca.review.interactive_plot import BASELINE_FUNCTIONS, _baseline_config_from_run_params

    params = dict(run_params or {})
    baseline_name, baseline_kwargs, warnings = _baseline_config_from_run_params(params)
    baseline_func = BASELINE_FUNCTIONS.get(baseline_name) or BASELINE_FUNCTIONS['per_camera_gp']
    kwargs = {
        'trigger_mode': str(params.get('trigger_mode') or TRIGGER_MODE),
        'logbf_threshold_dip': float(params.get('logbf_threshold_dip') or LOGBF_THRESHOLD_DIP),
        'logbf_threshold_jump': float(params.get('logbf_threshold_jump') or LOGBF_THRESHOLD_JUMP),
        'significance_threshold': float(params.get('significance_threshold') or SIGNIFICANCE_THRESHOLD),
        'p_points': int(params.get('p_points') or P_POINTS),
        'mag_points': int(params.get('mag_points') or MAG_POINTS),
        'run_min_points': int(params.get('run_min_points') or RUN_MIN_POINTS),
        'max_gap_points': int(params.get('run_max_gap_points') or RUN_MAX_GAP_POINTS),
        'run_max_gap_days': _finite_float(params.get('run_max_gap_days')),
        'run_min_duration_days': _finite_float(params.get('run_min_duration_days')),
        'compute_event_prob': True,
    }
    return baseline_func, baseline_kwargs, warnings, kwargs


def prepare_full_relative_flux_frame(candidate_id: str, payload: dict[str, object], controls: dict[str, object], *, lc_path: Path | None = None, run_params: dict | None = None) -> pd.DataFrame:
    from malca.review.interactive_plot import _baseline_config_from_run_params, _compute_baseline_bands

    df, resolved_lc_path = load_canonical_cleaned_lightcurve(payload, lc_path=lc_path, run_params=run_params)
    if df.empty:
        return pd.DataFrame()
    params = dict(run_params or {})
    baseline_name, baseline_kwargs, baseline_warnings = _baseline_config_from_run_params(params)
    cache_key = (str(resolved_lc_path.resolve()), 'adaptive_full_relative_flux')
    band_frames = _compute_baseline_bands(df, baseline_name, cache_key, baseline_kwargs=baseline_kwargs)
    merged = pd.concat([part.copy() for band, part in band_frames.items() if band in (0, 1)], ignore_index=True) if band_frames else pd.DataFrame()
    if merged.empty:
        return pd.DataFrame()
    if 'baseline' not in merged.columns or not np.isfinite(pd.to_numeric(merged['baseline'], errors='coerce')).any():
        merged['baseline'] = merged.groupby('v_g_band')['mag'].transform('median')
    work = merged.copy()
    for col in ('JD', 'mag', 'error', 'baseline'):
        work[col] = pd.to_numeric(work[col], errors='coerce')
    work = work[np.isfinite(work['JD']) & np.isfinite(work['mag']) & np.isfinite(work['error']) & (work['error'] > 0) & np.isfinite(work['baseline'])]
    work = work[work['v_g_band'].isin([0, 1])].copy()
    if work.empty:
        return pd.DataFrame()
    band_labels = {0: 'g', 1: 'V'}
    work['band'] = work['v_g_band'].map(lambda value: band_labels.get(int(value), str(value)))
    relative_flux = np.power(10.0, -0.4 * (work['mag'].to_numpy(dtype=float) - work['baseline'].to_numpy(dtype=float)))
    relative_flux_error = (math.log(10.0) / 2.5) * relative_flux * work['error'].to_numpy(dtype=float)
    start = _finite_float(controls.get('start_jd'))
    end = _finite_float(controls.get('end_jd'))
    t0 = _finite_float(controls.get('t0_jd'))
    if start is not None and end is not None and end < start:
        start, end = end, start
    in_window = np.ones(len(work), dtype=bool)
    if start is not None:
        in_window &= work['JD'].to_numpy(dtype=float) >= start
    if end is not None:
        in_window &= work['JD'].to_numpy(dtype=float) <= end
    out = pd.DataFrame(
        {
            'candidate_id': str(candidate_id),
            'time': work['JD'].to_numpy(dtype=float),
            'band': work['band'].astype(str).to_numpy(),
            'observed': relative_flux,
            'error': relative_flux_error,
            'source_mag': work['mag'].to_numpy(dtype=float),
            'baseline_mag': work['baseline'].to_numpy(dtype=float),
            'in_fit_window': in_window,
            'fit_start_jd': start,
            'fit_end_jd': end,
            'control_t0_jd': t0,
            'lc_path': str(resolved_lc_path),
            'baseline_name': baseline_name,
            'baseline_warnings': '; '.join(str(w) for w in baseline_warnings),
        }
    )
    return out[np.isfinite(out['time']) & np.isfinite(out['observed']) & np.isfinite(out['error']) & (out['error'] > 0)].reset_index(drop=True)


def _valid_lc_times(df: pd.DataFrame) -> np.ndarray:
    work = df.copy()
    if 'v_g_band' in work.columns:
        work = work[work['v_g_band'].isin([0, 1])]
    for col in ('JD', 'mag', 'error'):
        if col in work.columns:
            work[col] = pd.to_numeric(work[col], errors='coerce')
    work = work[np.isfinite(work.get('JD')) & np.isfinite(work.get('mag'))]
    if 'error' in work.columns:
        work = work[np.isfinite(work['error']) & (work['error'] > 0)]
    times = work['JD'].to_numpy(dtype=float)
    times = times[np.isfinite(times)]
    times.sort()
    return times


def _run_center(run: dict[str, object]) -> float | None:
    params = run.get('params') if isinstance(run.get('params'), dict) else {}
    center = _finite_float(params.get('t0'))
    if center is None:
        start = _finite_float(run.get('start_jd'))
        end = _finite_float(run.get('end_jd'))
        center = 0.5 * (start + end) if start is not None and end is not None else _finite_float(run.get('peak_jd'))
    return center


def _run_rank(run: dict[str, object], reference_t0: float | None) -> tuple[float, float, int, float]:
    center = _run_center(run)
    proximity = abs(center - reference_t0) if center is not None and reference_t0 is not None else 0.0
    return (
        _finite_float(run.get('run_sum'), -np.inf),
        _finite_float(run.get('run_max'), -np.inf),
        int(_finite_float(run.get('n_points'), 0) or 0),
        -float(proximity),
    )


def _select_adaptive_run(runs: list[dict[str, object]], reference_t0: float | None) -> tuple[dict[str, object] | None, str, list[str]]:
    warnings = []
    valid = [run for run in runs if _run_center(run) is not None]
    non_noise = [run for run in valid if str(run.get('morphology') or '').lower() != 'noise']
    if reference_t0 is not None:
        local = [run for run in non_noise if abs(float(_run_center(run)) - reference_t0) <= float(WINDOW_LOCAL_RADIUS_DAYS)]
    else:
        local = []
    if local:
        return max(local, key=lambda run: _run_rank(run, reference_t0)), 'adaptive_local_non_noise', warnings
    if reference_t0 is not None:
        warnings.append('no non-noise dip run within local radius')
    if non_noise:
        warnings.append('used strongest non-noise dip run outside local radius')
        return max(non_noise, key=lambda run: _run_rank(run, reference_t0)), 'adaptive_global_non_noise', warnings
    if valid:
        warnings.append('only noise/unclassified dip runs available')
        return max(valid, key=lambda run: _run_rank(run, reference_t0)), 'adaptive_global_any_run', warnings
    return None, 'adaptive_deepest_point_fallback', warnings


def _count_window_points(times: np.ndarray, start: float, end: float, event_start: float, event_end: float) -> tuple[int, int, int]:
    in_window = (times >= start) & (times <= end)
    pre = in_window & (times < event_start)
    post = in_window & (times > event_end)
    return int(np.count_nonzero(in_window)), int(np.count_nonzero(pre)), int(np.count_nonzero(post))


def _robust_baseline_flux(flux: np.ndarray) -> float | None:
    finite = np.asarray(flux, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return None
    lo, hi = np.nanpercentile(finite, [35, 95])
    trimmed = finite[(finite >= lo) & (finite <= hi)]
    if trimmed.size < 5:
        trimmed = finite
    return float(np.nanmedian(trimmed))


def _baseline_shoulder_window(
    full_lc: pd.DataFrame,
    center: float,
    event_start: float,
    event_end: float,
) -> tuple[float | None, float | None, dict[str, object]]:
    if full_lc.empty:
        return None, None, {'baseline_window_source': 'no_full_lc'}
    work = full_lc.copy()
    work['time'] = pd.to_numeric(work['time'], errors='coerce')
    work['observed'] = pd.to_numeric(work['observed'], errors='coerce')
    work = work[np.isfinite(work['time']) & np.isfinite(work['observed'])].sort_values('time')
    if work.empty:
        return None, None, {'baseline_window_source': 'no_valid_flux'}
    search_start = max(float(np.nanmin(work['time'])), center - float(WINDOW_MAX_HALF_WIDTH_DAYS))
    search_end = min(float(np.nanmax(work['time'])), center + float(WINDOW_MAX_HALF_WIDTH_DAYS))
    local = work[(work['time'] >= search_start) & (work['time'] <= search_end)].copy()
    if local.empty:
        local = work.copy()
    baseline = _robust_baseline_flux(local['observed'].to_numpy(dtype=float))
    if baseline is None:
        return None, None, {'baseline_window_source': 'no_baseline_estimate'}
    event_region = local[(local['time'] >= center - max(float(WINDOW_MIN_HALF_WIDTH_DAYS), 30.0)) & (local['time'] <= center + max(float(WINDOW_MIN_HALF_WIDTH_DAYS), 30.0))]
    if event_region.empty:
        event_region = local
    min_flux = float(np.nanmin(event_region['observed'].to_numpy(dtype=float)))
    depth = max(baseline - min_flux, float(WINDOW_BASELINE_FLUX_TOL))
    baseline_floor = baseline - max(float(WINDOW_BASELINE_FLUX_TOL), depth * float(WINDOW_BASELINE_RETURN_FRACTION))
    event_floor = baseline - max(float(WINDOW_BASELINE_FLUX_TOL), depth * float(WINDOW_EVENT_EXTENT_FRACTION))
    dimming = local[local['observed'] <= event_floor].sort_values('time')
    if not dimming.empty:
        dimming_times = dimming['time'].to_numpy(dtype=float)
        anchor = int(np.nanargmin(np.abs(dimming_times - center)))
        left = anchor
        right = anchor
        max_gap = float(WINDOW_EVENT_CLUSTER_GAP_DAYS)
        while left > 0 and dimming_times[left] - dimming_times[left - 1] <= max_gap:
            left -= 1
        while right + 1 < len(dimming_times) and dimming_times[right + 1] - dimming_times[right] <= max_gap:
            right += 1
        event_start = min(event_start, float(dimming_times[left]))
        event_end = max(event_end, float(dimming_times[right]))
    baseline_like = local[local['observed'] >= baseline_floor]
    left_baseline = baseline_like[baseline_like['time'] < event_start]
    right_baseline = baseline_like[baseline_like['time'] > event_end]
    n_required = int(WINDOW_BASELINE_SHOULDER_POINTS)
    left_ok = len(left_baseline) >= n_required
    right_ok = len(right_baseline) >= n_required
    start = None
    end = None
    if left_ok:
        start = float(left_baseline['time'].iloc[-n_required]) - float(WINDOW_SHOULDER_EXTRA_DAYS)
    if right_ok:
        end = float(right_baseline['time'].iloc[n_required - 1]) + float(WINDOW_SHOULDER_EXTRA_DAYS)
    if start is not None:
        start = max(start, center - float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmin(work['time'])))
    if end is not None:
        end = min(end, center + float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmax(work['time'])))
    if start is not None and end is not None and (end - start) < float(WINDOW_BASELINE_SHOULDER_MIN_DAYS):
        midpoint = 0.5 * (start + end)
        half = 0.5 * float(WINDOW_BASELINE_SHOULDER_MIN_DAYS)
        start = max(midpoint - half, center - float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmin(work['time'])))
        end = min(midpoint + half, center + float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmax(work['time'])))
    diagnostics = {
        'baseline_window_source': 'baseline_shoulders' if left_ok and right_ok else 'partial_baseline_shoulders',
        'baseline_flux': baseline,
        'baseline_floor': baseline_floor,
        'event_floor': event_floor,
        'baseline_shoulder_left_points': int(len(left_baseline)),
        'baseline_shoulder_right_points': int(len(right_baseline)),
        'baseline_event_start_jd': event_start,
        'baseline_event_end_jd': event_end,
    }
    return start, end, diagnostics


def _deepest_point_from_df(df: pd.DataFrame) -> float | None:
    work = df.copy()
    if 'v_g_band' in work.columns:
        work = work[work['v_g_band'].isin([0, 1])]
    if work.empty or 'JD' not in work.columns or 'mag' not in work.columns:
        return None
    work['JD'] = pd.to_numeric(work['JD'], errors='coerce')
    work['mag'] = pd.to_numeric(work['mag'], errors='coerce')
    work = work[np.isfinite(work['JD']) & np.isfinite(work['mag'])]
    if work.empty:
        return None
    return float(work.loc[work['mag'].idxmax(), 'JD'])


def adaptive_window_controls_for_candidate(
    conn,
    candidate_id: str,
    payload: dict[str, object],
    *,
    lc_path: Path | None = None,
    run_params: dict | None = None,
) -> tuple[dict[str, object], dict[str, object]]:
    original = control_defaults_for_candidate(conn, candidate_id, payload, lc_path=lc_path, run_params=run_params)
    if WINDOW_STRATEGY != 'adaptive_best_local':
        return dict(original), {'window_source': 'original_controls', 'warnings': ''}

    override = dict(WINDOW_OVERRIDES.get(str(candidate_id), {}) or {})
    df, _resolved = load_canonical_cleaned_lightcurve(payload, lc_path=lc_path, run_params=run_params)
    times = _valid_lc_times(df)
    warnings = []
    if times.size == 0:
        controls = dict(original)
        warnings.append('no valid cleaned g/V times; used original controls')
        return controls, {'window_source': 'original_controls_no_valid_times', 'warnings': '; '.join(warnings)}
    try:
        full_lc_for_window = prepare_full_relative_flux_frame(candidate_id, payload, original, lc_path=lc_path, run_params=run_params)
    except Exception as exc:
        full_lc_for_window = pd.DataFrame()
        warnings.append(f'could not build relative-flux shoulder frame: {exc}')

    base_t0 = _finite_float(original.get('t0_jd'))
    runs = []
    baseline_warnings = []
    if not override:
        try:
            baseline_func, baseline_kwargs, baseline_warnings, scoring_kwargs = _event_scoring_kwargs(run_params)
            score = score_lightcurve(df, baseline_func=baseline_func, baseline_kwargs=baseline_kwargs, **scoring_kwargs)
            runs = list((score.get('dip') or {}).get('run_summaries') or [])
        except Exception as exc:
            warnings.append(f'score_lightcurve failed: {exc}')

    selected_run, source, select_warnings = _select_adaptive_run(runs, base_t0)
    warnings.extend(str(w) for w in baseline_warnings)
    warnings.extend(select_warnings)

    if override:
        center = _finite_float(override.get('t0_jd'), base_t0)
        start = _finite_float(override.get('start_jd'))
        end = _finite_float(override.get('end_jd'))
        if center is None and start is not None and end is not None:
            center = 0.5 * (start + end)
        if center is None:
            center = _deepest_point_from_df(df)
        if start is None or end is None:
            half = max(float(WINDOW_MIN_HALF_WIDTH_DAYS), min(float(WINDOW_MAX_HALF_WIDTH_DAYS), float(WINDOW_PADDING_MAX_DAYS)))
            start, end = float(center - half), float(center + half)
        source = 'manual_override'
        event_start, event_end = start, end
        selected_morphology = 'manual'
        selected_run_sum = np.nan
        selected_run_max = np.nan
        selected_n_points = np.nan
    elif selected_run is not None:
        center = _run_center(selected_run)
        event_start = _finite_float(selected_run.get('start_jd'), center)
        event_end = _finite_float(selected_run.get('end_jd'), center)
        selected_morphology = str(selected_run.get('morphology') or 'unknown')
        selected_run_sum = _finite_float(selected_run.get('run_sum'))
        selected_run_max = _finite_float(selected_run.get('run_max'))
        selected_n_points = _finite_float(selected_run.get('n_points'))
        params = selected_run.get('params') if isinstance(selected_run.get('params'), dict) else {}
        sigma = abs(_finite_float(params.get('sigma'), 0.0) or 0.0)
        duration = abs(_finite_float(selected_run.get('duration_days'), 0.0) or 0.0)
        event_half = max(abs(center - event_start), abs(event_end - center), 0.5 * duration)
        if sigma > 0 and sigma <= float(WINDOW_MAX_HALF_WIDTH_DAYS):
            event_half = max(event_half, 3.0 * sigma)
        padding = min(max(float(WINDOW_PADDING_MIN_DAYS), event_half), float(WINDOW_PADDING_MAX_DAYS))
        half = min(max(float(WINDOW_MIN_HALF_WIDTH_DAYS), event_half + padding), float(WINDOW_MAX_HALF_WIDTH_DAYS))
        start, end = float(center - half), float(center + half)
    else:
        center = _deepest_point_from_df(df)
        if center is None:
            center = base_t0 if base_t0 is not None else float(np.nanmedian(times))
        event_start = center
        event_end = center
        selected_morphology = 'deepest_point'
        selected_run_sum = np.nan
        selected_run_max = np.nan
        selected_n_points = np.nan
        half = float(WINDOW_MIN_HALF_WIDTH_DAYS)
        start, end = float(center - half), float(center + half)
        warnings.append('no dip runs available; used deepest cleaned point')

    center = float(center)
    if event_end < event_start:
        event_start, event_end = event_end, event_start
    shoulder_start, shoulder_end, shoulder_diag = _baseline_shoulder_window(full_lc_for_window, center, event_start, event_end)
    shoulder_window_attempted = str(shoulder_diag.get('baseline_window_source') or '') in {'baseline_shoulders', 'partial_baseline_shoulders'}
    baseline_window_applied = shoulder_start is not None and shoulder_end is not None
    if shoulder_diag:
        event_start = _finite_float(shoulder_diag.get('baseline_event_start_jd'), event_start)
        event_end = _finite_float(shoulder_diag.get('baseline_event_end_jd'), event_end)
        if str(shoulder_diag.get('baseline_window_source') or '') == 'partial_baseline_shoulders':
            warnings.append(
                f"baseline shoulder search was partial: left={shoulder_diag.get('baseline_shoulder_left_points')}, right={shoulder_diag.get('baseline_shoulder_right_points')}"
            )
    if shoulder_start is not None:
        start = float(shoulder_start)
    else:
        warnings.append('no baseline-looking left shoulder found; using broad left padding')
        start = max(min(start, center - float(WINDOW_PADDING_MAX_DAYS)), center - float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmin(times)))
    if shoulder_end is not None:
        end = float(shoulder_end)
    else:
        warnings.append('no baseline-looking right shoulder found; using broad right padding')
        end = min(max(end, center + float(WINDOW_PADDING_MAX_DAYS)), center + float(WINDOW_MAX_HALF_WIDTH_DAYS), float(np.nanmax(times)))
    if baseline_window_applied:
        source = f'{source}_shouldered'
    guard_centers = []
    for run in runs:
        if str(run.get('morphology') or '').lower() == 'noise':
            continue
        run_center = _run_center(run)
        if run_center is not None and abs(float(run_center) - center) > WINDOW_MATCH_TOL_DAYS:
            guard_centers.append(float(run_center))
    run_centers = sorted(guard_centers)
    previous_centers = [c for c in run_centers if c < center - WINDOW_MATCH_TOL_DAYS]
    next_centers = [c for c in run_centers if c > center + WINDOW_MATCH_TOL_DAYS]
    data_min = float(np.nanmin(times))
    data_max = float(np.nanmax(times))
    if shoulder_window_attempted:
        lower_guard = max(data_min, float(start))
        upper_guard = min(data_max, float(end))
    else:
        lower_guard = max(data_min, 0.5 * (previous_centers[-1] + center) if previous_centers else data_min)
        upper_guard = min(data_max, 0.5 * (next_centers[0] + center) if next_centers else data_max)
    guard_clipped = False

    half = max(abs(center - start), abs(end - center), float(WINDOW_MIN_HALF_WIDTH_DAYS))
    initial_half = half
    best = None
    while True:
        candidate_start = max(center - half, lower_guard)
        candidate_end = min(center + half, upper_guard)
        guard_clipped = guard_clipped or candidate_start > center - half + WINDOW_MATCH_TOL_DAYS or candidate_end < center + half - WINDOW_MATCH_TOL_DAYS
        total, pre, post = _count_window_points(times, candidate_start, candidate_end, event_start, event_end)
        best = (candidate_start, candidate_end, total, pre, post, half)
        if total >= int(WINDOW_MIN_POINTS) and pre >= int(WINDOW_MIN_SIDE_POINTS) and post >= int(WINDOW_MIN_SIDE_POINTS):
            break
        if half >= float(WINDOW_MAX_HALF_WIDTH_DAYS) - WINDOW_MATCH_TOL_DAYS:
            break
        half = min(float(WINDOW_MAX_HALF_WIDTH_DAYS), max(half + 7.0, half * 1.35))

    start, end, total, pre, post, half = best
    support_ok = total >= int(WINDOW_MIN_POINTS) and pre >= int(WINDOW_MIN_SIDE_POINTS) and post >= int(WINDOW_MIN_SIDE_POINTS)
    if guard_clipped and not support_ok and not shoulder_window_attempted:
        relaxed_half = initial_half
        relaxed_best = None
        while True:
            candidate_start = max(center - relaxed_half, data_min)
            candidate_end = min(center + relaxed_half, data_max)
            relaxed_total, relaxed_pre, relaxed_post = _count_window_points(times, candidate_start, candidate_end, event_start, event_end)
            relaxed_best = (candidate_start, candidate_end, relaxed_total, relaxed_pre, relaxed_post, relaxed_half)
            if relaxed_total >= int(WINDOW_MIN_POINTS) and relaxed_pre >= int(WINDOW_MIN_SIDE_POINTS) and relaxed_post >= int(WINDOW_MIN_SIDE_POINTS):
                break
            if relaxed_half >= float(WINDOW_MAX_HALF_WIDTH_DAYS) - WINDOW_MATCH_TOL_DAYS:
                break
            relaxed_half = min(float(WINDOW_MAX_HALF_WIDTH_DAYS), max(relaxed_half + 7.0, relaxed_half * 1.35))
        if relaxed_best is not None and relaxed_best[2] > total:
            start, end, total, pre, post, half = relaxed_best
            warnings.append('relaxed neighboring-dip guard to meet point support')
        else:
            warnings.append('window clipped to avoid neighboring dip center')
    elif guard_clipped and not shoulder_window_attempted:
        warnings.append('window clipped to avoid neighboring dip center')
    if total < int(WINDOW_MIN_POINTS):
        warnings.append(f'adaptive window has {total} points < {WINDOW_MIN_POINTS}')
    if pre < int(WINDOW_MIN_SIDE_POINTS) or post < int(WINDOW_MIN_SIDE_POINTS):
        warnings.append(f'adaptive side counts pre={pre}, post={post}')
    if base_t0 is not None and abs(center - base_t0) > float(WINDOW_LOCAL_RADIUS_DAYS):
        warnings.append('adaptive center is outside local radius from original t0')

    controls = dict(original)
    controls.update(
        {
            't0_jd': float(center),
            'start_jd': float(start),
            'end_jd': float(end),
            'half_width_days': float(max(center - start, end - center)),
            'window_strategy': WINDOW_STRATEGY,
            'window_source': source,
            'message': f"Adaptive DustyCult window ({source}). " + str(original.get('message') or ''),
        }
    )
    diagnostic = {
        'window_source': source,
        'selected_morphology': selected_morphology,
        'selected_run_sum': selected_run_sum,
        'selected_run_max': selected_run_max,
        'selected_n_points': selected_n_points,
        'selected_event_start_jd': event_start,
        'selected_event_end_jd': event_end,
        'adaptive_side_pre_points': pre,
        'adaptive_side_post_points': post,
        'adaptive_total_points_estimate': total,
        'baseline_window_source': shoulder_diag.get('baseline_window_source'),
        'baseline_flux': shoulder_diag.get('baseline_flux'),
        'baseline_floor': shoulder_diag.get('baseline_floor'),
        'event_floor': shoulder_diag.get('event_floor'),
        'baseline_shoulder_left_points': shoulder_diag.get('baseline_shoulder_left_points'),
        'baseline_shoulder_right_points': shoulder_diag.get('baseline_shoulder_right_points'),
        'neighbor_lower_guard_jd': lower_guard,
        'neighbor_upper_guard_jd': upper_guard,
        'warnings': '; '.join(str(w) for w in warnings if str(w).strip()),
    }
    return controls, diagnostic


def _prepared_input_count(payload: dict[str, object], controls: dict[str, object], *, lc_path: Path | None = None, run_params: dict | None = None) -> tuple[int, str]:
    try:
        prepared = prepare_dustycult_input(payload, controls, lc_path=lc_path, run_params=run_params)
        return int(len(prepared.frame)), ''
    except Exception as exc:
        return 0, str(exc)


fit_candidates = selected_candidate_frame()
adaptive_controls_by_candidate = {}
window_diagnostic_rows = []
adaptive_full_lc_frames = []

with db_connect(DB_PATH) as conn:
    for _, row in fit_candidates.iterrows():
        candidate_id = str(row['candidate_id'])
        payload = row_payload(row)
        lc_path = resolve_candidate_lc_path(row, payload)
        if lc_path is not None:
            payload['lc_path'] = str(lc_path)
        original_controls = control_defaults_for_candidate(conn, candidate_id, payload, lc_path=lc_path, run_params=RUN_PARAMS)
        adaptive_controls, diagnostic = adaptive_window_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path, run_params=RUN_PARAMS)
        original_n, original_error = _prepared_input_count(payload, original_controls, lc_path=lc_path, run_params=RUN_PARAMS)
        adaptive_n, adaptive_error = _prepared_input_count(payload, adaptive_controls, lc_path=lc_path, run_params=RUN_PARAMS)
        if adaptive_n < int(WINDOW_MIN_POINTS) and original_n >= int(WINDOW_MIN_POINTS):
            fallback_warning = f'adaptive window had {adaptive_n} points; kept original supported window with {original_n} points'
            existing_warnings = str(diagnostic.get('warnings') or '')
            diagnostic = dict(diagnostic)
            diagnostic['window_source'] = 'original_min_support_fallback'
            diagnostic['warnings'] = '; '.join(part for part in (existing_warnings, fallback_warning) if part)
            adaptive_controls = dict(original_controls)
            adaptive_controls.update(
                {
                    'window_strategy': WINDOW_STRATEGY,
                    'window_source': 'original_min_support_fallback',
                    'message': f'Kept original DustyCult window because adaptive support was too sparse. ' + str(original_controls.get('message') or ''),
                }
            )
            adaptive_n, adaptive_error = original_n, original_error
        adaptive_controls_by_candidate[candidate_id] = adaptive_controls
        full_lc = prepare_full_relative_flux_frame(candidate_id, payload, adaptive_controls, lc_path=lc_path, run_params=RUN_PARAMS)
        if not full_lc.empty:
            adaptive_full_lc_frames.append(full_lc)
        original_start = _finite_float(original_controls.get('start_jd'))
        original_end = _finite_float(original_controls.get('end_jd'))
        adaptive_start = _finite_float(adaptive_controls.get('start_jd'))
        adaptive_end = _finite_float(adaptive_controls.get('end_jd'))
        changed = not (
            original_start is not None
            and original_end is not None
            and adaptive_start is not None
            and adaptive_end is not None
            and abs(original_start - adaptive_start) <= WINDOW_MATCH_TOL_DAYS
            and abs(original_end - adaptive_end) <= WINDOW_MATCH_TOL_DAYS
            and abs(_finite_float(original_controls.get('t0_jd'), np.nan) - _finite_float(adaptive_controls.get('t0_jd'), np.nan)) <= WINDOW_MATCH_TOL_DAYS
        )
        window_diagnostic_rows.append(
            {
                'candidate_id': candidate_id,
                'window_changed': changed,
                'original_source': original_controls.get('source'),
                'adaptive_source': diagnostic.get('window_source'),
                'original_t0_jd': original_controls.get('t0_jd'),
                'adaptive_t0_jd': adaptive_controls.get('t0_jd'),
                'original_start_jd': original_start,
                'original_end_jd': original_end,
                'adaptive_start_jd': adaptive_start,
                'adaptive_end_jd': adaptive_end,
                'original_window_days': (original_end - original_start) if original_start is not None and original_end is not None else np.nan,
                'adaptive_window_days': (adaptive_end - adaptive_start) if adaptive_start is not None and adaptive_end is not None else np.nan,
                'original_n_input_points': original_n,
                'adaptive_n_input_points': adaptive_n,
                'selected_morphology': diagnostic.get('selected_morphology'),
                'selected_run_sum': diagnostic.get('selected_run_sum'),
                'selected_run_max': diagnostic.get('selected_run_max'),
                'selected_n_points': diagnostic.get('selected_n_points'),
                'adaptive_side_pre_points': diagnostic.get('adaptive_side_pre_points'),
                'adaptive_side_post_points': diagnostic.get('adaptive_side_post_points'),
                'baseline_window_source': diagnostic.get('baseline_window_source'),
                'baseline_shoulder_left_points': diagnostic.get('baseline_shoulder_left_points'),
                'baseline_shoulder_right_points': diagnostic.get('baseline_shoulder_right_points'),
                'original_input_error': original_error,
                'adaptive_input_error': adaptive_error,
                'warnings': diagnostic.get('warnings') or '',
            }
        )

adaptive_window_df = pd.DataFrame(window_diagnostic_rows)
adaptive_full_lc_df = pd.concat(adaptive_full_lc_frames, ignore_index=True) if adaptive_full_lc_frames else pd.DataFrame()

if len(adaptive_window_df) != len(fit_candidates):
    raise AssertionError(f'Expected {len(fit_candidates)} adaptive windows, found {len(adaptive_window_df)}')

print(f'Adaptive windows built: {len(adaptive_window_df)}')
display(
    adaptive_window_df[
        [
            'candidate_id',
            'window_changed',
            'adaptive_source',
            'original_window_days',
            'adaptive_window_days',
            'original_n_input_points',
            'adaptive_n_input_points',
            'selected_morphology',
            'selected_run_sum',
            'baseline_window_source',
            'baseline_shoulder_left_points',
            'baseline_shoulder_right_points',
            'adaptive_side_pre_points',
            'adaptive_side_post_points',
            'warnings',
        ]
    ]
)
display(adaptive_window_df.groupby(['adaptive_source', 'window_changed'], dropna=False).size().reset_index(name='n'))


def plot_window_diagnostic(candidate_id: str) -> go.Figure:
    if 'adaptive_window_df' not in globals() or adaptive_window_df.empty:
        raise ValueError('Run the Window Diagnostics cell first.')
    row = adaptive_window_df[adaptive_window_df['candidate_id'].astype(str).eq(str(candidate_id))]
    if row.empty:
        raise ValueError(f'No adaptive window row found for {candidate_id}')
    row = row.iloc[-1]
    full_lc = adaptive_full_lc_df[adaptive_full_lc_df['candidate_id'].astype(str).eq(str(candidate_id))].copy()
    if full_lc.empty:
        raise ValueError(f'No full light-curve rows found for {candidate_id}')

    fig = go.Figure()
    colors = {'g': '#2a9d55', 'V': '#3d6fb6'}
    original_start = _finite_float(row.get('original_start_jd'))
    original_end = _finite_float(row.get('original_end_jd'))
    adaptive_start = _finite_float(row.get('adaptive_start_jd'))
    adaptive_end = _finite_float(row.get('adaptive_end_jd'))
    original_t0 = _finite_float(row.get('original_t0_jd'))
    adaptive_t0 = _finite_float(row.get('adaptive_t0_jd'))

    if original_start is not None and original_end is not None:
        fig.add_vrect(x0=original_start, x1=original_end, fillcolor='rgba(210, 80, 70, 0.07)', line_width=1, line_dash='dot', line_color='rgba(210,80,70,0.75)', layer='below')
    if adaptive_start is not None and adaptive_end is not None:
        fig.add_vrect(x0=adaptive_start, x1=adaptive_end, fillcolor='rgba(245, 200, 80, 0.16)', line_width=0, layer='below')
    if original_t0 is not None:
        fig.add_vline(x=original_t0, line={'color': 'rgba(210,80,70,0.75)', 'width': 1, 'dash': 'dot'}, annotation_text='orig t0', annotation_position='top left')
    if adaptive_t0 is not None:
        fig.add_vline(x=adaptive_t0, line={'color': 'rgba(30,30,30,0.75)', 'width': 1.5, 'dash': 'dash'}, annotation_text='adaptive t0', annotation_position='top right')

    for band, part in full_lc.sort_values('time').groupby('band'):
        color = colors.get(str(band), None)
        in_window = part['in_fit_window'].astype(bool) if 'in_fit_window' in part.columns else pd.Series(True, index=part.index)
        outside = part.loc[~in_window]
        inside = part.loc[in_window]
        if not outside.empty:
            fig.add_trace(go.Scatter(x=outside['time'], y=outside['observed'], mode='markers', name=f'{band} outside adaptive', marker={'size': 4, 'color': color, 'opacity': 0.28}, error_y={'type': 'data', 'array': outside['error'], 'visible': True, 'thickness': 0.6}))
        if not inside.empty:
            fig.add_trace(go.Scatter(x=inside['time'], y=inside['observed'], mode='markers', name=f'{band} inside adaptive', marker={'size': 6, 'color': color, 'opacity': 0.9}, error_y={'type': 'data', 'array': inside['error'], 'visible': True}))
    fig.update_layout(
        title=f"Adaptive DustyCult window: {candidate_id} ({row.get('adaptive_source')}, n={row.get('adaptive_n_input_points')})",
        xaxis_title='JD',
        yaxis_title='Relative flux',
        template='plotly_white',
        height=560,
    )
    return fig


window_plot_candidates = fit_candidates['candidate_id'].astype(str).tolist()
if MAX_FIT_PLOTS is not None:
    window_plot_candidates = window_plot_candidates[:int(MAX_FIT_PLOTS)]
for candidate_id in window_plot_candidates:
    display(Markdown(f'### {candidate_id}'))
    display(plot_window_diagnostic(candidate_id))


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Adaptive windows built: 21


,candidate_id,window_changed,adaptive_source,original_window_days,adaptive_window_days,original_n_input_points,adaptive_n_input_points,selected_morphology,selected_run_sum,baseline_window_source,baseline_shoulder_left_points,baseline_shoulder_right_points,adaptive_side_pre_points,adaptive_side_post_points,warnings
0,111669557747,True,adaptive_local_non_noise,240.000000,586.257171,61,132,gaussian,67.000000,partial_baseline_shoulders,2,31,2,21,"baseline shoulder search was partial: left=2, ..."
1,146029419304,True,adaptive_local_non_noise,14.000000,328.206077,4,109,gaussian,3.000000,partial_baseline_shoulders,4,110,0,23,"baseline shoulder search was partial: left=4, ..."
2,180388903123,True,adaptive_local_non_noise,144.781180,608.026794,11,153,gaussian,44.999999,partial_baseline_shoulders,78,9,42,9,"baseline shoulder search was partial: left=78,..."
3,206158525635,True,adaptive_deepest_point_fallback_shouldered,240.000000,530.266840,279,342,deepest_point,NaN,baseline_shoulders,12,140,14,33,no non-noise dip run within local radius; no d...
4,223338997633,True,adaptive_local_non_noise_shouldered,14.000000,559.282950,10,169,gaussian,3.000000,baseline_shoulders,67,44,13,36,
5,240518636016,True,adaptive_local_non_noise_shouldered,41.808217,326.016540,10,41,gaussian,4.000000,baseline_shoulders,32,30,11,18,
6,249109084130,True,adaptive_deepest_point_fallback_shouldered,14.000000,518.583770,9,284,deepest_point,NaN,baseline_shoulders,36,102,19,15,no non-noise dip run within local radius; no d...
7,25771086021,True,adaptive_local_non_noise_shouldered,71.582180,388.115940,18,91,gaussian,2.000000,baseline_shoulders,63,15,17,13,
8,369367489518,True,adaptive_local_non_noise,48.117790,242.068500,23,144,gaussian,19.000000,partial_baseline_shoulders,4,89,1,14,"baseline shoulder search was partial: left=4, ..."
9,446676921101,True,adaptive_local_non_noise_shouldered,163.941920,336.888470,23,62,gaussian,9.000000,baseline_shoulders,87,60,21,18,


,adaptive_source,window_changed,n
0,adaptive_deepest_point_fallback_shouldered,True,2
1,adaptive_global_non_noise_shouldered,True,1
2,adaptive_local_non_noise,True,7
3,adaptive_local_non_noise_shouldered,True,11


### 111669557747

...

### 592705518006

### 644245286164

### 661425468910

### 8591303502

### 94489786439

## Dry-Run Queue

Inspect this table before launching fits. `full` is planned only when quick is already `ok` or quick is also planned.

In [7]:
if 'fit_candidates' not in globals():
    fit_candidates = selected_candidate_frame()
require_adaptive_windows_ready(fit_candidates['candidate_id'].astype(str).tolist())

queue_rows = []
window_lookup = adaptive_window_df.set_index('candidate_id') if 'adaptive_window_df' in globals() and not adaptive_window_df.empty else pd.DataFrame()

for _, row in fit_candidates.iterrows():
    candidate_id = str(row['candidate_id'])
    payload = row_payload(row)
    lc_path = resolve_candidate_lc_path(row, payload)
    with db_connect(DB_PATH) as conn:
        controls = active_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path, run_params=RUN_PARAMS)
    quick_status = mode_status(existing_fits, candidate_id, 'quick')
    full_status = mode_status(existing_fits, candidate_id, 'full')
    quick_action = planned_mode_action(existing_fits, candidate_id, 'quick', controls)
    quick_will_run = 'quick' in MODES and quick_action != 'skipped_existing_ok'
    quick_available_for_full = quick_status == 'ok' or quick_will_run
    full_action = planned_mode_action(existing_fits, candidate_id, 'full', controls)
    full_will_run = 'full' in MODES and quick_available_for_full and full_action != 'skipped_existing_ok'
    win = window_lookup.loc[candidate_id].to_dict() if not window_lookup.empty and candidate_id in window_lookup.index else {}
    queue_rows.append(
        {
            'candidate_id': candidate_id,
            'lc_found': lc_path is not None,
            'lc_path': str(lc_path) if lc_path else '',
            'window_changed': win.get('window_changed'),
            'adaptive_source': win.get('adaptive_source'),
            'adaptive_window_days': win.get('adaptive_window_days'),
            'adaptive_n_input_points': win.get('adaptive_n_input_points'),
            'quick_status': quick_status or 'missing',
            'quick_action': quick_action if 'quick' in MODES else 'disabled',
            'quick_window_matches': fit_window_matches(existing_fits, candidate_id, 'quick', controls),
            'full_status': full_status or 'missing',
            'full_action': full_action if full_will_run else ('skipped_existing_ok' if full_action == 'skipped_existing_ok' else 'wait_for_quick'),
            'full_window_matches': fit_window_matches(existing_fits, candidate_id, 'full', controls),
            'dipper_score': row.get('dipper_score'),
            'dip_run_count': row.get('dip_run_count'),
        }
    )

run_queue = pd.DataFrame(queue_rows)
display(run_queue)
display(run_queue.groupby(['quick_action', 'full_action'], dropna=False).size().reset_index(name='n'))


,candidate_id,lc_found,lc_path,window_changed,adaptive_source,adaptive_window_days,adaptive_n_input_points,quick_status,quick_action,quick_window_matches,full_status,full_action,full_window_matches,dipper_score,dip_run_count
0,111669557747,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise,586.257171,132,ok,rerun_window_changed,False,missing,run_missing_or_failed,False,19.914586,3.0
1,146029419304,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise,328.206077,109,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,21.255792,3.0
2,180388903123,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise,608.026794,153,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,-1.040463,4.0
3,206158525635,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_deepest_point_fallback_shouldered,530.266840,342,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,20.882040,1.0
4,223338997633,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise_shouldered,559.282950,169,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,19.852360,3.0
5,240518636016,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise_shouldered,326.016540,41,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,19.800610,2.0
6,249109084130,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_deepest_point_fallback_shouldered,518.583770,284,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,0.000000,0.0
7,25771086021,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise_shouldered,388.115940,91,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,15.589737,1.0
8,369367489518,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise,242.068500,144,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,21.709102,5.0
9,446676921101,True,/Users/calder/code/malca/output/runs/runs_marc...,True,adaptive_local_non_noise_shouldered,336.888470,62,missing,run_missing_or_failed,False,missing,run_missing_or_failed,False,-0.560155,1.0


,quick_action,full_action,n
0,rerun_window_changed,run_missing_or_failed,2
1,run_missing_or_failed,run_missing_or_failed,19


## Deterministic DustyCult Prefits

This diagnostic-only pass fits a DustyCult-like forward model with bounded least squares before running MCMC. It writes nothing to the review DB; results live in `deterministic_results_df` and `deterministic_curves_df`.

In [8]:
def _finite_float(value, default=None):
    try:
        if value is None or pd.isna(value):
            return default
    except Exception:
        if value is None:
            return default
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if math.isfinite(number) else default


def _deterministic_stellar_grid(star_R: float, star_u1: float, star_u2: float, grid_n: int) -> dict[str, np.ndarray | float]:
    R = max(float(star_R), 1e-6)
    xs = np.linspace(-R, R, int(grid_n))
    ys = np.linspace(-R, R, int(grid_n))
    dx = xs[1] - xs[0] if len(xs) > 1 else 2 * R
    dy = ys[1] - ys[0] if len(ys) > 1 else 2 * R
    xx, yy = np.meshgrid(xs, ys)
    r2 = xx * xx + yy * yy
    inside = r2 <= R * R
    mu = np.zeros_like(xx, dtype=float)
    mu[inside] = np.sqrt(np.maximum(0.0, 1.0 - r2[inside] / (R * R)))
    intensity = 1.0 - float(star_u1) * (1.0 - mu) - float(star_u2) * (1.0 - mu) ** 2
    intensity[~inside] = 0.0
    weights = np.full_like(intensity, dx * dy, dtype=float)
    f0 = float(np.sum(intensity * weights))
    if not math.isfinite(f0) or f0 <= 0:
        raise ValueError('Invalid deterministic stellar grid normalization.')
    return {
        'x': xx.ravel(),
        'y': yy.ravel(),
        'intensity': intensity.ravel(),
        'weights': weights.ravel(),
        'f0': f0,
    }


def _theta_to_dust_params(theta: np.ndarray, lambda0: float) -> dict[str, float]:
    return {
        't0': float(theta[0]),
        'v': float(np.exp(theta[1])),
        'b': float(theta[2]),
        'tau0': float(np.exp(theta[3])),
        'lambda0': float(lambda0),
        'alpha': float(theta[4]),
        'sigma_y': float(np.exp(theta[5])),
        'sigma_x_plus': float(np.exp(theta[6])),
        'sigma_x_minus': float(np.exp(theta[7])),
    }


def _deterministic_flux(theta: np.ndarray, times: np.ndarray, wavelengths: np.ndarray, grid: dict[str, np.ndarray | float], lambda0: float) -> np.ndarray:
    dust = _theta_to_dust_params(theta, lambda0)
    x_center = dust['v'] * (times[:, None] - dust['t0'])
    y_center = dust['b']
    x_occ = grid['x'][None, :] - x_center
    y_occ = grid['y'][None, :] - y_center
    gaussian_y = np.exp(-0.5 * (y_occ / dust['sigma_y']) ** 2)
    shape_plus = np.exp(-0.5 * (x_occ / dust['sigma_x_plus']) ** 2) * gaussian_y
    shape_minus = np.exp(-0.5 * (x_occ / dust['sigma_x_minus']) ** 2) * gaussian_y
    shape = 0.5 * (shape_plus + shape_minus)
    opacity = dust['tau0'] * np.power(wavelengths[:, None] / dust['lambda0'], -dust['alpha'])
    tau = shape * opacity
    transmission = np.exp(-tau)
    weighted = grid['intensity'][None, :] * transmission * grid['weights'][None, :]
    return np.sum(weighted, axis=1) / float(grid['f0'])


def _deterministic_initial_guess(frame: pd.DataFrame, controls: dict[str, object]) -> np.ndarray:
    times = frame['time'].to_numpy(dtype=float)
    flux = frame['relative_flux'].to_numpy(dtype=float)
    baseline = float(np.nanmedian(flux))
    dip_index = int(np.nanargmin(flux))
    depth = max(baseline - float(flux[dip_index]), 1e-4)
    threshold = baseline - depth / 2.0
    in_dip = np.isfinite(times) & np.isfinite(flux) & (flux <= threshold)
    if np.count_nonzero(in_dip) >= 2:
        duration = float(np.nanmax(times[in_dip]) - np.nanmin(times[in_dip]))
    else:
        duration = max(float(np.nanmax(times) - np.nanmin(times)) / 5.0, 1.0)
    duration = max(duration, 1e-3)
    star_R = max(_finite_float(controls.get('star_R'), 1.0), 1e-6)
    sigma = max(0.25 * star_R, 1e-3)
    return np.array(
        [
            _finite_float(controls.get('t0_jd'), float(times[dip_index])),
            math.log(max(2.0 / duration, 1e-4)),
            _finite_float(controls.get('b_center'), 0.0),
            math.log(depth),
            _finite_float(controls.get('alpha_center'), 0.0),
            math.log(sigma),
            math.log(sigma),
            math.log(sigma),
        ],
        dtype=float,
    )


def _deterministic_bounds(frame: pd.DataFrame, controls: dict[str, object]) -> tuple[np.ndarray, np.ndarray]:
    times = frame['time'].to_numpy(dtype=float)
    start = _finite_float(controls.get('start_jd'), float(np.nanmin(times)))
    end = _finite_float(controls.get('end_jd'), float(np.nanmax(times)))
    if end < start:
        start, end = end, start
    star_R = max(_finite_float(controls.get('star_R'), 1.0), 1e-6)
    sigma_min = max(0.01 * star_R, 1e-3)
    sigma_max = max(4.0 * star_R, 0.1)
    b_range = max(2.0 * star_R, 2.0)
    lower = np.array([start, math.log(1e-4), -b_range, math.log(1e-5), -6.0, math.log(sigma_min), math.log(sigma_min), math.log(sigma_min)], dtype=float)
    upper = np.array([end, math.log(50.0), b_range, math.log(10.0), 6.0, math.log(sigma_max), math.log(sigma_max), math.log(sigma_max)], dtype=float)
    return lower, upper


def prepare_full_relative_flux_frame(candidate_id: str, payload: dict[str, object], controls: dict[str, object], *, lc_path: Path | None = None, run_params: dict | None = None) -> pd.DataFrame:
    from malca.review.interactive_plot import _baseline_config_from_run_params, _compute_baseline_bands

    df, resolved_lc_path = load_canonical_cleaned_lightcurve(payload, lc_path=lc_path, run_params=run_params)
    if df.empty:
        return pd.DataFrame()
    params = dict(run_params or {})
    baseline_name, baseline_kwargs, baseline_warnings = _baseline_config_from_run_params(params)
    cache_key = (str(resolved_lc_path.resolve()), 'deterministic_full_relative_flux')
    band_frames = _compute_baseline_bands(df, baseline_name, cache_key, baseline_kwargs=baseline_kwargs)
    merged = pd.concat([part.copy() for band, part in band_frames.items() if band in (0, 1)], ignore_index=True) if band_frames else pd.DataFrame()
    if merged.empty:
        return pd.DataFrame()
    if 'baseline' not in merged.columns or not np.isfinite(pd.to_numeric(merged['baseline'], errors='coerce')).any():
        merged['baseline'] = merged.groupby('v_g_band')['mag'].transform('median')
    work = merged.copy()
    for col in ('JD', 'mag', 'error', 'baseline'):
        work[col] = pd.to_numeric(work[col], errors='coerce')
    work = work[np.isfinite(work['JD']) & np.isfinite(work['mag']) & np.isfinite(work['error']) & (work['error'] > 0) & np.isfinite(work['baseline'])]
    work = work[work['v_g_band'].isin([0, 1])].copy()
    if work.empty:
        return pd.DataFrame()
    band_labels = {0: 'g', 1: 'V'}
    work['band'] = work['v_g_band'].map(lambda value: band_labels.get(int(value), str(value)))
    relative_flux = np.power(10.0, -0.4 * (work['mag'].to_numpy(dtype=float) - work['baseline'].to_numpy(dtype=float)))
    relative_flux_error = (math.log(10.0) / 2.5) * relative_flux * work['error'].to_numpy(dtype=float)
    start = _finite_float(controls.get('start_jd'))
    end = _finite_float(controls.get('end_jd'))
    t0 = _finite_float(controls.get('t0_jd'))
    if start is not None and end is not None and end < start:
        start, end = end, start
    in_window = np.ones(len(work), dtype=bool)
    if start is not None:
        in_window &= work['JD'].to_numpy(dtype=float) >= start
    if end is not None:
        in_window &= work['JD'].to_numpy(dtype=float) <= end
    out = pd.DataFrame(
        {
            'candidate_id': str(candidate_id),
            'time': work['JD'].to_numpy(dtype=float),
            'band': work['band'].astype(str).to_numpy(),
            'observed': relative_flux,
            'error': relative_flux_error,
            'source_mag': work['mag'].to_numpy(dtype=float),
            'baseline_mag': work['baseline'].to_numpy(dtype=float),
            'in_fit_window': in_window,
            'fit_start_jd': start,
            'fit_end_jd': end,
            'control_t0_jd': t0,
            'lc_path': str(resolved_lc_path),
            'baseline_name': baseline_name,
            'baseline_warnings': '; '.join(str(w) for w in baseline_warnings),
        }
    )
    return out[np.isfinite(out['time']) & np.isfinite(out['observed']) & np.isfinite(out['error']) & (out['error'] > 0)].reset_index(drop=True)


def fit_deterministic_prefit(candidate_id: str, prepared, controls: dict[str, object]) -> tuple[dict[str, object], pd.DataFrame]:
    frame = prepared.frame.copy()
    wavelengths = frame['band'].map(DUSTYCULT_BANDPASS_NM).astype(float).to_numpy()
    if not np.isfinite(wavelengths).all():
        raise ValueError('Deterministic prefit only supports bands in DUSTYCULT_BANDPASS_NM.')
    times = frame['time'].to_numpy(dtype=float)
    observed = frame['relative_flux'].to_numpy(dtype=float)
    errors = frame['relative_flux_error'].to_numpy(dtype=float)
    lambda0 = float(np.nanmedian(wavelengths))
    star_R = max(_finite_float(controls.get('star_R'), 1.0), 1e-6)
    star_u1 = _finite_float(controls.get('star_u1'), 0.0)
    star_u2 = _finite_float(controls.get('star_u2'), 0.0)
    grid = _deterministic_stellar_grid(star_R, star_u1, star_u2, DETERMINISTIC_GRID_N)
    lower, upper = _deterministic_bounds(frame, controls)
    theta0 = np.clip(_deterministic_initial_guess(frame, controls), lower + 1e-9, upper - 1e-9)

    def residual(theta):
        model = _deterministic_flux(theta, times, wavelengths, grid, lambda0)
        return (model - observed) / errors

    started = time.monotonic()
    result = least_squares(
        residual,
        theta0,
        bounds=(lower, upper),
        loss='soft_l1',
        f_scale=1.0,
        max_nfev=int(DETERMINISTIC_MAX_NFEV),
        x_scale='jac',
    )
    runtime = time.monotonic() - started
    model = _deterministic_flux(result.x, times, wavelengths, grid, lambda0)
    residuals = (model - observed) / errors
    chi2 = float(np.sum(residuals ** 2))
    dof = max(int(len(observed) - len(result.x)), 1)
    reduced_chi2 = chi2 / dof
    params = _theta_to_dust_params(result.x, lambda0)
    status = 'ok' if result.success and math.isfinite(reduced_chi2) else 'failed'
    row = {
        'candidate_id': str(candidate_id),
        'status': status,
        'runtime_sec': runtime,
        'n_input_points': int(len(frame)),
        'nfev': int(result.nfev),
        'cost': float(result.cost),
        'chi2': chi2,
        'reduced_chi2': reduced_chi2,
        'lambda0': params['lambda0'],
        't0': params['t0'],
        'v': params['v'],
        'b': params['b'],
        'tau0': params['tau0'],
        'alpha': params['alpha'],
        'sigma_y': params['sigma_y'],
        'sigma_x_plus': params['sigma_x_plus'],
        'sigma_x_minus': params['sigma_x_minus'],
        'message': str(result.message),
    }
    curves = pd.DataFrame(
        {
            'candidate_id': str(candidate_id),
            'time': times,
            'band': frame['band'].astype(str).to_numpy(),
            'observed': observed,
            'error': errors,
            'model': model,
            'residual_sigma': residuals,
        }
    )
    return row, curves


deterministic_results = []
deterministic_curve_frames = []
deterministic_full_lc_frames = []

require_adaptive_windows_ready(fit_candidates['candidate_id'].astype(str).tolist())

if RUN_DETERMINISTIC_PREFIT:
    with db_connect(DB_PATH) as conn:
        for idx, row in fit_candidates.iterrows():
            candidate_id = str(row['candidate_id'])
            payload = row_payload(row)
            lc_path = resolve_candidate_lc_path(row, payload)
            if lc_path is not None:
                payload['lc_path'] = str(lc_path)
            print(f'[{idx + 1}/{len(fit_candidates)}] deterministic {candidate_id}')
            try:
                controls = active_controls_for_candidate(
                    conn,
                    candidate_id,
                    payload,
                    lc_path=lc_path,
                    run_params=RUN_PARAMS,
                )
                prepared = prepare_dustycult_input(
                    payload,
                    controls,
                    lc_path=lc_path,
                    run_params=RUN_PARAMS,
                )
                fit_row, curves = fit_deterministic_prefit(candidate_id, prepared, controls)
                full_lc = prepare_full_relative_flux_frame(candidate_id, payload, controls, lc_path=lc_path, run_params=RUN_PARAMS)
                deterministic_results.append(fit_row)
                deterministic_curve_frames.append(curves)
                if not full_lc.empty:
                    deterministic_full_lc_frames.append(full_lc)
                print(f"  deterministic: {fit_row['status']} chi2nu={fit_row['reduced_chi2']:.3g} nfev={fit_row['nfev']}")
            except Exception as exc:
                deterministic_results.append({'candidate_id': candidate_id, 'status': 'failed', 'error': str(exc)})
                print(f'  deterministic: failed {exc}')

deterministic_results_df = pd.DataFrame(deterministic_results)
deterministic_curves_df = pd.concat(deterministic_curve_frames, ignore_index=True) if deterministic_curve_frames else pd.DataFrame()
deterministic_full_lc_df = pd.concat(deterministic_full_lc_frames, ignore_index=True) if deterministic_full_lc_frames else pd.DataFrame()

if not deterministic_results_df.empty:
    display(deterministic_results_df.sort_values(['status', 'reduced_chi2'], na_position='last'))
    display(deterministic_results_df.groupby('status', dropna=False).size().reset_index(name='n'))
else:
    display(Markdown('Deterministic prefit is disabled or produced no rows.'))

if EXPORT_DETERMINISTIC_RESULTS and not deterministic_results_df.empty:
    export_path = RUN_DIR / 'review' / 'dustycult' / 'deterministic_prefits.csv'
    deterministic_results_df.to_csv(export_path, index=False)
    print(f'Wrote {export_path}')


[1/21] deterministic 111669557747
  deterministic: ok chi2nu=7.29 nfev=359
[2/21] deterministic 146029419304
  deterministic: ok chi2nu=19.9 nfev=47
[3/21] deterministic 180388903123
  deterministic: ok chi2nu=1.71 nfev=204
[4/21] deterministic 206158525635
  deterministic: ok chi2nu=20.7 nfev=114
[5/21] deterministic 223338997633
  deterministic: ok chi2nu=36.3 nfev=74
[6/21] deterministic 240518636016
  deterministic: ok chi2nu=22.6 nfev=88
[7/21] deterministic 249109084130
  deterministic: ok chi2nu=12.5 nfev=278
[8/21] deterministic 25771086021
  deterministic: failed chi2nu=11.7 nfev=400
[9/21] deterministic 369367489518
  deterministic: failed chi2nu=185 nfev=400
[10/21] deterministic 446676921101
  deterministic: failed chi2nu=5.99 nfev=400
[11/21] deterministic 481036586933
  deterministic: ok chi2nu=19.4 nfev=112
[12/21] deterministic 498216222923
  deterministic: failed chi2nu=2.9 nfev=400
[13/21] deterministic 515396131751
  deterministic: failed chi2nu=55.1 nfev=400
[14/21]

,candidate_id,status,runtime_sec,n_input_points,nfev,cost,chi2,reduced_chi2,lambda0,t0,v,b,tau0,alpha,sigma_y,sigma_x_plus,sigma_x_minus,message
11,498216222923,failed,11.678176,134,400,91.113188,365.750336,2.902780,477.0,8956.691382,0.131107,-0.047528,5.734025,1.126561e+00,2.596125,6.817329,6.817234,The maximum number of function evaluations is ...
19,8591303502,failed,15.420761,196,400,177.524443,707.701193,3.764368,477.0,8315.415194,0.028982,-8.524780,4.085956,-2.699265e+00,3.097079,0.758879,0.758874,The maximum number of function evaluations is ...
15,549755992463,failed,13.665668,162,400,177.506014,873.468155,5.671871,477.0,10020.474542,0.597480,-15.354530,1.008867,3.347073e+00,5.874771,5.341604,41.397886,The maximum number of function evaluations is ...
9,446676921101,failed,4.045722,62,400,76.904859,323.679499,5.994065,477.0,8892.322139,0.312402,-18.195922,9.991599,5.940000e+00,3.171171,5.632551,5.632550,The maximum number of function evaluations is ...
14,532576054353,failed,26.594828,323,400,479.516448,2825.634116,8.970267,477.0,8775.599657,0.563370,0.007795,4.668794,-2.146804e-16,0.694691,0.454442,0.255511,The maximum number of function evaluations is ...
7,25771086021,failed,6.874645,91,400,154.608541,968.136932,11.664300,477.0,8874.400735,0.169608,-0.008616,9.619293,-1.679437e+00,3.965199,1.013838,1.013838,The maximum number of function evaluations is ...
12,515396131751,failed,3.156966,47,400,207.097230,2147.983373,55.076497,477.0,11095.486046,0.077818,0.051951,9.860496,2.837158e-16,0.208667,0.211788,0.211788,The maximum number of function evaluations is ...
8,369367489518,failed,12.210789,144,400,1409.853434,25207.744437,185.351062,477.0,9455.348660,0.022603,-0.003900,7.020132,-6.658746e-16,0.419525,0.118056,0.118056,The maximum number of function evaluations is ...
16,592705518006,failed,7.999283,113,400,1201.403717,32702.874290,311.455946,477.0,9520.685112,0.106286,-0.011281,9.998201,5.727354e+00,4.426258,1.497242,1.497242,The maximum number of function evaluations is ...
2,180388903123,ok,6.528961,153,204,76.558092,247.686053,1.708180,477.0,8746.077893,0.004107,0.006221,9.999814,4.956017e+00,0.623362,0.029452,0.033589,`ftol` termination condition is satisfied.


,status,n
0,failed,9
1,ok,12


## Deterministic Prefit Plots

These plots are available immediately after the deterministic prefit cell. They show the full cleaned relative-flux light curve with faint points outside the fit window, a shaded fit window, and t0 markers; they do not require quick or full DustyCult MCMC fits to finish.

In [9]:
def plot_deterministic_prefit(candidate_id: str) -> go.Figure:
    if 'deterministic_curves_df' not in globals() or deterministic_curves_df.empty:
        raise ValueError('Run the deterministic prefit cell first.')
    fit_curves = deterministic_curves_df[deterministic_curves_df['candidate_id'].astype(str).eq(str(candidate_id))].copy()
    if fit_curves.empty:
        raise ValueError(f'No deterministic curve rows found for {candidate_id}')
    full_lc = pd.DataFrame()
    if 'deterministic_full_lc_df' in globals() and not deterministic_full_lc_df.empty:
        full_lc = deterministic_full_lc_df[deterministic_full_lc_df['candidate_id'].astype(str).eq(str(candidate_id))].copy()
    if full_lc.empty and 'adaptive_full_lc_df' in globals() and not adaptive_full_lc_df.empty:
        full_lc = adaptive_full_lc_df[adaptive_full_lc_df['candidate_id'].astype(str).eq(str(candidate_id))].copy()
    observed_curves = full_lc if not full_lc.empty else fit_curves.assign(in_fit_window=True)

    chi2_text = ''
    fitted_t0 = None
    if 'deterministic_results_df' in globals() and not deterministic_results_df.empty:
        row = deterministic_results_df[deterministic_results_df['candidate_id'].astype(str).eq(str(candidate_id))]
        if not row.empty:
            if 'reduced_chi2' in row.columns:
                chi2 = _finite_float(row.iloc[-1].get('reduced_chi2'))
                if chi2 is not None:
                    chi2_text = f' chi2nu={chi2:.3g}'
            fitted_t0 = _finite_float(row.iloc[-1].get('t0'))
    fit_start = _finite_float(observed_curves['fit_start_jd'].dropna().iloc[0]) if 'fit_start_jd' in observed_curves and observed_curves['fit_start_jd'].notna().any() else float(fit_curves['time'].min())
    fit_end = _finite_float(observed_curves['fit_end_jd'].dropna().iloc[0]) if 'fit_end_jd' in observed_curves and observed_curves['fit_end_jd'].notna().any() else float(fit_curves['time'].max())
    control_t0 = _finite_float(observed_curves['control_t0_jd'].dropna().iloc[0]) if 'control_t0_jd' in observed_curves and observed_curves['control_t0_jd'].notna().any() else None

    fig = go.Figure()
    colors = {'g': '#2a9d55', 'V': '#3d6fb6'}
    if fit_start is not None and fit_end is not None:
        fig.add_vrect(x0=fit_start, x1=fit_end, fillcolor='rgba(245, 200, 80, 0.13)', line_width=0, layer='below')
    if control_t0 is not None:
        fig.add_vline(x=control_t0, line={'color': 'rgba(120,120,120,0.55)', 'width': 1, 'dash': 'dot'}, annotation_text='control t0', annotation_position='top left')
    if fitted_t0 is not None:
        fig.add_vline(x=fitted_t0, line={'color': 'rgba(30,30,30,0.75)', 'width': 1.5, 'dash': 'dash'}, annotation_text='fit t0', annotation_position='top right')

    for band, part in observed_curves.sort_values('time').groupby('band'):
        color = colors.get(str(band), None)
        in_window = part['in_fit_window'].astype(bool) if 'in_fit_window' in part.columns else pd.Series(True, index=part.index)
        outside = part.loc[~in_window]
        inside = part.loc[in_window]
        if not outside.empty:
            fig.add_trace(
                go.Scatter(
                    x=outside['time'],
                    y=outside['observed'],
                    mode='markers',
                    name=f'{band} observed outside window',
                    marker={'size': 4, 'color': color, 'opacity': 0.28},
                    error_y={'type': 'data', 'array': outside['error'], 'visible': True, 'thickness': 0.6},
                )
            )
        if not inside.empty:
            fig.add_trace(
                go.Scatter(
                    x=inside['time'],
                    y=inside['observed'],
                    mode='markers',
                    name=f'{band} observed fit window',
                    marker={'size': 6, 'color': color, 'opacity': 0.9},
                    error_y={'type': 'data', 'array': inside['error'], 'visible': True},
                )
            )
    for band, part in fit_curves.sort_values('time').groupby('band'):
        color = colors.get(str(band), None)
        fig.add_trace(
            go.Scatter(
                x=part['time'],
                y=part['model'],
                mode='lines',
                name=f'{band} deterministic model',
                line={'width': 2, 'dash': 'dash', 'color': color},
            )
        )
    fig.update_layout(
        title=f'Deterministic DustyCult prefit, full light curve: {candidate_id}{chi2_text}',
        xaxis_title='JD',
        yaxis_title='Relative flux',
        template='plotly_white',
        height=560,
    )
    return fig


det_plot_candidates = fit_candidates['candidate_id'].astype(str).tolist() if 'fit_candidates' in globals() else reviewed_dippers['candidate_id'].astype(str).tolist()
if MAX_FIT_PLOTS is not None:
    det_plot_candidates = det_plot_candidates[:int(MAX_FIT_PLOTS)]

det_plot_status_rows = []
for candidate_id in det_plot_candidates:
    has_deterministic = bool(
        'deterministic_curves_df' in globals()
        and not deterministic_curves_df.empty
        and deterministic_curves_df['candidate_id'].astype(str).eq(str(candidate_id)).any()
    )
    det_status = 'missing'
    if 'deterministic_results_df' in globals() and not deterministic_results_df.empty:
        det_row = deterministic_results_df[deterministic_results_df['candidate_id'].astype(str).eq(str(candidate_id))]
        if not det_row.empty:
            det_status = str(det_row.iloc[-1].get('status') or 'unknown')
    det_plot_status_rows.append({'candidate_id': candidate_id, 'deterministic_status': det_status, 'has_deterministic_curves': has_deterministic})
    if has_deterministic:
        display(Markdown(f'### {candidate_id}'))
        display(plot_deterministic_prefit(candidate_id))

deterministic_plot_status_df = pd.DataFrame(det_plot_status_rows)
display(deterministic_plot_status_df)


### 111669557747

### 146029419304

### 180388903123

### 206158525635

### 223338997633

...

### 644245286164

### 661425468910

### 8591303502

### 94489786439

,candidate_id,deterministic_status,has_deterministic_curves
0,111669557747,ok,True
1,146029419304,ok,True
2,180388903123,ok,True
3,206158525635,ok,True
4,223338997633,ok,True
5,240518636016,ok,True
6,249109084130,ok,True
7,25771086021,failed,True
8,369367489518,failed,True
9,446676921101,failed,True


## Run DustyCult Fits

Running this cell writes to the review DB and creates/replaces artifacts for modes that are not skipped. For a smoke test, set `MAX_CANDIDATES = 1` above and rerun the configuration, load, helper, and dry-run cells first.

In [ ]:
run_results = []
require_adaptive_windows_ready(fit_candidates['candidate_id'].astype(str).tolist())


def record_result(candidate_id: str, mode: str, action: str, row: dict[str, object] | None = None, error: str = '') -> None:
    row = dict(row or {})
    run_results.append(
        {
            'candidate_id': str(candidate_id),
            'mode': mode,
            'action': action,
            'status': row.get('status', ''),
            'runtime_sec': row.get('runtime_sec'),
            't0_jd': row.get('t0_jd'),
            'start_jd': row.get('start_jd'),
            'end_jd': row.get('end_jd'),
            'n_input_points': row.get('n_input_points'),
            'n_curve_points': row.get('n_curve_points'),
            'artifact_dir': row.get('artifact_dir', ''),
            'error': row.get('error', error),
        }
    )


with db_connect(DB_PATH) as conn:
    for idx, row in fit_candidates.iterrows():
        candidate_id = str(row['candidate_id'])
        payload = row_payload(row)
        lc_path = resolve_candidate_lc_path(row, payload)
        if lc_path is not None:
            payload['lc_path'] = str(lc_path)

        print(f'[{idx + 1}/{len(fit_candidates)}] {candidate_id}')
        fits_before = load_dustycult_fits(conn, candidate_id)
        controls = active_controls_for_candidate(
            conn,
            candidate_id,
            payload,
            lc_path=lc_path,
            run_params=RUN_PARAMS,
        )

        if 'quick' in MODES:
            quick_action = planned_mode_action(fits_before, candidate_id, 'quick', controls)
            if quick_action == 'skipped_existing_ok':
                record_result(candidate_id, 'quick', quick_action, {'status': 'ok', **controls})
                print('  quick: skipped existing ok with matching window')
            else:
                quick_row = run_dustycult_fit(
                    conn,
                    candidate_id,
                    payload,
                    db_path=DB_PATH,
                    controls=controls,
                    mode='quick',
                    lc_path=lc_path,
                    run_params=RUN_PARAMS,
                    project_path=DUSTYCULT_PROJECT,
                    julia=JULIA,
                )
                record_result(candidate_id, 'quick', quick_action, quick_row)
                print(f"  quick: {quick_row.get('status')} {quick_action} {quick_row.get('error') or ''}")

        fits_after_quick = load_dustycult_fits(conn, candidate_id)
        quick_ok = mode_status(fits_after_quick, candidate_id, 'quick') == 'ok'

        if 'full' in MODES:
            if not quick_ok:
                record_result(candidate_id, 'full', 'skipped_quick_not_ok', {'status': 'skipped', **controls}, error='Quick fit is not ok.')
                print('  full: skipped because quick is not ok')
            else:
                full_action = planned_mode_action(fits_after_quick, candidate_id, 'full', controls)
                if full_action == 'skipped_existing_ok':
                    record_result(candidate_id, 'full', full_action, {'status': 'ok', **controls})
                    print('  full: skipped existing ok with matching window')
                else:
                    full_row = run_dustycult_fit(
                        conn,
                        candidate_id,
                        payload,
                        db_path=DB_PATH,
                        controls=controls,
                        mode='full',
                        lc_path=lc_path,
                        run_params=RUN_PARAMS,
                        project_path=DUSTYCULT_PROJECT,
                        julia=JULIA,
                    )
                    record_result(candidate_id, 'full', full_action, full_row)
                    print(f"  full: {full_row.get('status')} {full_action} {full_row.get('error') or ''}")

run_results_df = pd.DataFrame(run_results)
display(run_results_df)
display(run_results_df.groupby(['mode', 'action', 'status'], dropna=False).size().reset_index(name='n'))


[1/21] 111669557747
  quick: ok rerun_window_changed 


## Plot Helpers

Use these after at least one successful fit. These helpers reuse the same pure DustyCult display logic as the MALCA Review app: fit status, posterior predictive plot, circumstellar geometry, occulter visualization, and posterior parameter summaries.


In [ ]:
def _display_rows_table(rows, columns=None):
    if not rows:
        display(Markdown('_No rows available._'))
        return
    if columns is None:
        columns = ['Field', 'Value']
    display(pd.DataFrame(rows, columns=columns))


def _load_selected_dustycult_fit(candidate_id: str, mode: str | None = None):
    with db_connect(DB_PATH) as conn:
        fits = load_dustycult_fits(conn, candidate_id)
        fit_row = select_dustycult_display_row(fits, mode=mode)
        if fit_row is None:
            raise ValueError(f'No DustyCult fit found for {candidate_id}')
        selected_mode = str(fit_row.get('mode') or mode or 'quick')
        curves = load_dustycult_curve(conn, candidate_id, selected_mode)
    return fit_row, curves


def plot_dustycult_fit(candidate_id: str, mode: str | None = None) -> go.Figure:
    fit_row, curves = _load_selected_dustycult_fit(candidate_id, mode)
    return build_dustycult_fit_figure(curves, fit_row, theme='white')


def plot_dustycult_occulter(candidate_id: str, mode: str | None = None) -> go.Figure:
    fit_row, _curves = _load_selected_dustycult_fit(candidate_id, mode)
    if str(fit_row.get('status') or '').lower() != 'ok':
        raise ValueError(f'No successful DustyCult fit found for {candidate_id}')
    return build_dustycult_occulter_figure(fit_row, theme='white', grid_n=501)


def display_dustycult_review_panel(candidate_id: str, mode: str | None = None) -> None:
    fit_row, curves = _load_selected_dustycult_fit(candidate_id, mode)
    selected_mode = str(fit_row.get('mode') or mode or 'quick')
    status = str(fit_row.get('status') or 'unknown')
    display(Markdown(f'### {candidate_id} - DustyCult {selected_mode} ({status})'))
    _display_rows_table(dustycult_fit_metadata_rows(fit_row))
    if status.lower() == 'ok':
        display(build_dustycult_fit_figure(curves, fit_row, theme='white'))
        try:
            display(build_dustycult_occulter_figure(fit_row, theme='white', grid_n=501))
        except Exception as exc:
            display(Markdown(f'**Occulter model unavailable:** `{exc}`'))
    else:
        display(Markdown(f'**DustyCult fit is not OK:** `{fit_row.get("error") or status}`'))
    display(Markdown('#### Circumstellar Dust Geometry'))
    try:
        _display_rows_table(dustycult_geometry_rows(fit_row))
    except Exception as exc:
        display(Markdown(f'_Geometry unavailable: `{exc}`_'))
    display(Markdown('#### Posterior Parameters'))
    posterior_rows = dustycult_posterior_rows(fit_row, limit=None)
    _display_rows_table(posterior_rows, columns=['Parameter', 'Median', 'p16', 'p84'])


## Full Fit Review Panels

Run this after the full DustyCult fits finish. Each displayed panel mirrors the DustyCult presentation in MALCA Review, including the circumstellar geometry and posterior parameter summary.


In [ ]:
full_plot_candidates = fit_candidates['candidate_id'].astype(str).tolist() if 'fit_candidates' in globals() else reviewed_dippers['candidate_id'].astype(str).tolist()
if MAX_FIT_PLOTS is not None:
    full_plot_candidates = full_plot_candidates[:int(MAX_FIT_PLOTS)]

full_plot_status_rows = []
for candidate_id in full_plot_candidates:
    with db_connect(DB_PATH) as conn:
        full_fits = load_dustycult_fits(conn, candidate_id)
        full_status = mode_status(full_fits, candidate_id, FULL_PLOT_MODE)
        full_curves = load_dustycult_curve(conn, candidate_id, FULL_PLOT_MODE) if full_status == 'ok' else pd.DataFrame()
    has_full = not full_curves.empty
    full_plot_status_rows.append(
        {
            'candidate_id': candidate_id,
            f'{FULL_PLOT_MODE}_status': full_status or 'missing',
            f'has_{FULL_PLOT_MODE}_curves': has_full,
        }
    )
    if has_full:
        display_dustycult_review_panel(candidate_id, FULL_PLOT_MODE)

full_plot_status_df = pd.DataFrame(full_plot_status_rows)
display(full_plot_status_df)


## Example Plots

This picks the first successful full fit if one exists, otherwise the first successful quick fit.

In [ ]:
if 'final_fits' not in globals():
    candidate_ids = tuple(reviewed_dippers['candidate_id'].astype(str)) if 'reviewed_dippers' in globals() else tuple()
    placeholders = ','.join(['?'] * len(candidate_ids))
    with db_connect(DB_PATH) as conn:
        final_fits = pd.read_sql_query(
            f'SELECT * FROM dustycult_fits WHERE candidate_id IN ({placeholders})' if candidate_ids else 'SELECT * FROM dustycult_fits WHERE 0',
            conn,
            params=candidate_ids,
        )

successful = final_fits[final_fits['status'].astype(str).eq('ok')].copy()
if successful.empty:
    display(Markdown('No successful DustyCult fits are available yet.'))
else:
    successful['mode_rank'] = successful['mode'].map({'full': 0, 'quick': 1}).fillna(2)
    example = successful.sort_values(['mode_rank', 'candidate_id']).iloc[0]
    example_id = str(example['candidate_id'])
    example_mode = str(example['mode'])
    display(Markdown(f'Example candidate: `{example_id}` mode `{example_mode}`'))
    display_dustycult_review_panel(example_id, example_mode)
